In [9]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import glob
import os
import seaborn as sns
import plotly as pltx
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
# Set the folder path 
folder_path = "VDS2526_Madrid"

# Get all madrid_YYYY.csv files (excluding stations.csv)
csv_files = glob.glob(os.path.join(folder_path, "madrid_*.csv"))

dfs = []
for file in sorted(csv_files):
    # Extract the year from the filename, e.g. "madrid_2001.csv" -> "2001"
    year = os.path.splitext(os.path.basename(file))[0].split("_")[-1]
    
    df = pd.read_csv(file)
    df["year"] = year  # or int(year) if you want it as an integer
    dfs.append(df)

# Combine all into one DataFrame
df = pd.concat(dfs, ignore_index=True)

print(df.shape)
#print(df.head())
print(df["year"].value_counts().sort_index())

(3808224, 20)
year
2001    217872
2002    217296
2003    243984
2004    245496
2005    237000
2006    230568
2007    225120
2008    226392
2009    215688
2010    209448
2011    209928
2012    210720
2013    209880
2014    210024
2015    210096
2016    209496
2017    210120
2018     69096
Name: count, dtype: int64


In [13]:
# The stations file contains the name, lat and lon of the stations
stations = pd.read_csv(os.path.join(folder_path, "stations.csv"))
stations

,id,name,address,lon,lat,elevation
0,28079004,Pza. de España,Plaza de España,-3.712247,40.423853,635
1,28079008,Escuelas Aguirre,Entre C/ Alcalá y C/ O’ Donell,-3.682319,40.421564,670
2,28079011,Avda. Ramón y Cajal,Avda. Ramón y Cajal esq. C/ Príncipe de Vergara,-3.677356,40.451475,708
3,28079016,Arturo Soria,C/ Arturo Soria esq. C/ Vizconde de los Asilos,-3.639233,40.440047,693
4,28079017,Villaverde,C/. Juan Peñalver,-3.713322,40.347139,604
5,28079018,Farolillo,Calle Farolillo - C/Ervigio,-3.731853,40.394781,630
6,28079024,Casa de Campo,Casa de Campo (Terminal del Teleférico),-3.747347,40.419356,642
7,28079027,Barajas Pueblo,"C/. Júpiter, 21 (Barajas)",-3.580031,40.476928,621
8,28079035,Pza. del Carmen,Plaza del Carmen esq. Tres Cruces.,-3.703172,40.419208,659
9,28079036,Moratalaz,Avd. Moratalaz esq. Camino de los Vinateros,-3.645306,40.407947,685


In [5]:
# Data Management

# date
df['date'] = pd.to_datetime(df['date'])

df['year']  = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day']   = df['date'].dt.day
df['hour']  = df['date'].dt.hour

# Merge Station info
df = df.merge(
    stations[['id', 'name', 'lat', 'lon']],
    left_on='station',
    right_on='id',
    how='left'
)

df.head(5)

,date,BEN,CO,EBE,MXY,NMHC,NO_2,NOx,OXY,O_3,...,PM25,NO,CH4,month,day,hour,id,name,lat,lon
0,2001-08-01 01:00:00,NaN,0.37,NaN,NaN,NaN,58.400002,87.150002,NaN,34.529999,...,NaN,NaN,NaN,8,1,1,NaN,NaN,NaN,NaN
1,2001-08-01 01:00:00,1.5,0.34,1.49,4.1,0.07,56.250000,75.169998,2.11,42.160000,...,NaN,NaN,NaN,8,1,1,28079035.0,Pza. del Carmen,40.419208,-3.703172
2,2001-08-01 01:00:00,NaN,0.28,NaN,NaN,NaN,50.660000,61.380001,NaN,46.310001,...,NaN,NaN,NaN,8,1,1,NaN,NaN,NaN,NaN
3,2001-08-01 01:00:00,NaN,0.47,NaN,NaN,NaN,69.790001,73.449997,NaN,40.650002,...,NaN,NaN,NaN,8,1,1,28079004.0,Pza. de España,40.423853,-3.712247
4,2001-08-01 01:00:00,NaN,0.39,NaN,NaN,NaN,22.830000,24.799999,NaN,66.309998,...,NaN,NaN,NaN,8,1,1,28079039.0,Barrio del Pilar,40.478228,-3.711542


In [6]:
# Total stations on file 
datastn = set(df['station'].unique())

# Total stations info given
filestn = set(stations['id'].unique())

print(f'data stations - {len(datastn)}')
print(f'file stations - {len(filestn)}')


print(f'data + file stations - {len(datastn & filestn)}')
print(f'data + file stations - {len(datastn - filestn)}')

data stations - 39
file stations - 24
data + file stations - 24
data + file stations - 15


### Observation: 15 stations do not have a lat / lon / name. 

In [7]:
# Check missing values in each column
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_summary = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
}).sort_values('missing_pct', ascending=False)

print(missing_summary[missing_summary['missing_count'] > 0])

      missing_count  missing_pct
CH4         3793374    99.610054
MXY         3492809    91.717530
PXY         3492640    91.713093
OXY         3492529    91.710178
PM25        2991800    78.561555
EBE         2806500    73.695770
TOL         2769295    72.718805
BEN         2766540    72.646462
NMHC        2722912    71.500836
TCH         2721783    71.471190
NO          2275827    59.760849
NOx         1431949    37.601491
CO          1157212    30.387183
lat         1146240    30.099070
name        1146240    30.099070
lon         1146240    30.099070
id          1146240    30.099070
SO_2        1032264    27.106179
PM10         946969    24.866421
O_3          816492    21.440230
NO_2          21174     0.556007


In [15]:
# Data Wrangling to get the common stations and pollutants with less than 50% missingness of data

# Pollutants based on the WHO list, excluding PM25 as missingness is > 75% 
WHO = ['PM10', 'O_3', 'NO_2', 'SO_2', 'CO']

# Year wise missingness by Station
df_2001 = df[df['year']==2001].groupby('station').agg(lambda x: (x.isnull().sum()/len(x))*100)[WHO]
df_2018 = df[df['year']==2018].groupby('station').agg(lambda x: (x.isnull().sum()/len(x))*100)[WHO]

# Stations that qualify
station_dict = {}
for year, dataframe in [('2001', df_2001), ('2018', df_2018)]:
    for pollutant in WHO:
        stations_list = set(dataframe[dataframe[pollutant] <= 50].index.tolist())
        station_dict[(year, pollutant)] = stations_list
        
# Common Stations
common_stations = {}
for pollutant in df_2001.columns:
    stations_2001 = set(station_dict[('2001', pollutant)])
    stations_2018 = set(station_dict[('2018', pollutant)])
    common_stations[pollutant] = sorted(stations_2001 & stations_2018)

# print it
for pollutant, stations_name in common_stations.items():
    print(f"{pollutant}: {stations_name}")
    
# set id as index to match station index in avg dfs
stations_new = stations.set_index('id')
pollutants = list(common_stations.keys())

# join metadata
avg_2001 = (df[df['year'] == 2001]
            .groupby('station')[pollutants]
            .mean()
            .join(stations_new[['name', 'lat', 'lon']]))

avg_2018 = (df[df['year'] == 2018]
            .groupby('station')[pollutants]
            .mean()
            .join(stations_new[['name', 'lat', 'lon']]))

PM10: [28079008, 28079018, 28079024, 28079036, 28079038, 28079040]
O_3: [28079008, 28079016, 28079017, 28079018, 28079024, 28079035, 28079039]
NO_2: [28079004, 28079008, 28079011, 28079016, 28079017, 28079018, 28079024, 28079035, 28079036, 28079038, 28079039, 28079040]
SO_2: [28079004, 28079008, 28079017, 28079018, 28079024, 28079035, 28079036, 28079038, 28079040]
CO: [28079004, 28079008, 28079016, 28079018, 28079024, 28079035, 28079036, 28079039]


In [16]:
def get_plot_data(pollutant):
    stations_list = common_stations[pollutant]
    x_2001 = avg_2001.loc[stations_list, pollutant]
    x_2018 = avg_2018.loc[stations_list, pollutant]
    diff = x_2018 - x_2001
    names = avg_2018.loc[stations_list, 'name']
    lats = avg_2018.loc[stations_list, 'lat']
    lons = avg_2018.loc[stations_list, 'lon']
    return stations_list, x_2001, x_2018, diff, names, lats, lons

In [39]:
fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.65, 0.35],
    subplot_titles=("Average Value in 2018", "Diverging Bar: Change 2001→2018"),
    specs=[[{"type": "mapbox"}, {"type": "xy"}]]  # ← mapbox here
)

for i, pollutant in enumerate(pollutants):
    stations_list, x_2001, x_2018, diff, names, lats, lons = get_plot_data(pollutant)
    scaled_size = (x_2018 / x_2018.max()) * 50
    visible = (i == 0)

    # sort by diff ascending so highest change at top
    sorted_idx = diff.sort_values(ascending=True).index
    sorted_diff = diff.loc[sorted_idx]
    sorted_names = names.loc[sorted_idx]
    sorted_2018 = x_2018.loc[sorted_idx]

    fig.add_trace(go.Scattermapbox(        # ← Scattermapbox instead of Scattermap
        lat=lats, lon=lons,
        mode='markers',
        text=names,
        hovertemplate='<b>%{text}</b><br>Mean 2018: %{marker.color:.2f}<extra></extra>',
        visible=visible,
        marker=go.scattermapbox.Marker(    # ← use the explicit Marker class
            size=scaled_size,              # your normalized 0–50 values
            color=x_2018,
            colorscale='Reds',
            showscale=False,
            cmin=x_2018.min(),
            cmax=x_2018.max(),
            opacity=0.8
        )
    ))
    fig.add_trace(go.Bar(
        x=sorted_diff,
        y=list(range(len(sorted_names))),   # ← numbers instead of names
        orientation='h',
        visible=visible,
        text=[f'{v:.1f}' for v in sorted_diff],
        textposition='outside',
        textfont=dict(size=10),
        customdata=sorted_names,            # ← names go here for hover
        hovertemplate='<b>%{customdata}</b><br>Change: %{x:.2f}<extra></extra>',
        marker=dict(
            color=sorted_2018,
            colorscale='Reds',
            showscale=True,
            colorbar=dict(x=1.02, len=0.9),
            cmin=x_2018.min(),
            cmax=x_2018.max()
        )
    ), row=1, col=2)

# --- Dropdown ---
buttons = []
for i, pollutant in enumerate(pollutants):
    visibility = [False] * (len(pollutants) * 2)
    visibility[i * 2] = True
    visibility[i * 2 + 1] = True
    buttons.append(dict(
            label=pollutant,
    method='update',
    args=[
        {'visible': visibility},
        {'title': dict(             # ← dict not string
            text=f'{pollutant}: Top Polluted Areas in Madrid 2018 & Change 2001→2018',
            x=0.05,
            xanchor='left'
        )}
    ]
    ))

fig.update_layout(
    mapbox=dict(
        style='open-street-map',
        center=dict(lat=40.42, lon=-3.70),
        zoom=11,
        domain=dict(x=[0, 0.65], y=[0, 1])  # constrains map to left subplot
    ),
    updatemenus=[dict(
        buttons=buttons,
        direction='down',
        x=0.95, y=1.32,        # move dropdown to top right, away from title
        xanchor='right',
        showactive=True
    )],
    title=dict(
        text=f'{pollutants[0]}: Top Polluted Areas in Madrid 2018 & Change 2001→2018',
        x=0.05,                # left align title so dropdown never overlaps
        xanchor='left'
    ),

    showlegend=False,
    height=550,
    plot_bgcolor='lightgrey',
    margin=dict(l=50, r=100, t=80, b=100),
)

fig.add_annotation(
    text="<b>Pollutant</b>",
    x=0.8, y=1.3,    # ← move x to the left
    xref="paper", yref="paper",
    showarrow=False,
    font=dict(size=13)
)

fig.update_layout(
    yaxis=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=False
    )
)

# --- Axis labels ---

fig.update_xaxes(
    title_text="Change (2018 - 2001)",
    title_standoff=30,                 # pushes title down away from annotations
    zeroline=True,
    zerolinecolor='red',
    zerolinewidth=2,
    #showticklabels=False,   # ← hides station name labels
    automargin=True,
    row=1, col=2
)

fig.show(renderer="iframe")
fig.write_html("madrid_pollution.html")

/var/folders/9c/ry43vzsn0rx31lrth82xg43h0000gr/T/ipykernel_90984/3530261727.py:19: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(        # ← Scattermapbox instead of Scattermap
